# Sales Analyst 1.5B Fine-Tune (QLoRA)

## What this notebook does

This notebook fine-tunes a small (1.5B parameter) code-generation model —
`Qwen/Qwen2.5-Coder-1.5B-Instruct` — to answer natural-language sales
questions by writing short Python scripts against the `salestools` library
(e.g. "Is my revenue trending up?" → a snippet that calls
`decompose_trend(...)`). It uses QLoRA (a 4-bit quantized base model plus a
small trainable LoRA adapter) via the Unsloth library, which makes
fine-tuning fast and memory-efficient enough to run on a single Colab GPU.

Pipeline: clone this repo → synthesize + verify a training dataset of
question/code pairs → load the base model in 4-bit → attach LoRA adapters →
fine-tune → save and download the adapter (a few hundred MB — not the full
model).

## Run this in Google Colab

This notebook is designed to run in **Google Colab**, not locally — it
depends on a Colab GPU runtime (tested on an NVIDIA L4) and installs
GPU-specific packages (`unsloth`, and `bitsandbytes` via its extras) that
assume a CUDA environment. To run it:

1. Open this notebook in Colab.
2. Runtime → Change runtime type → GPU (L4 or better recommended).
3. Run cells top-to-bottom, in order — later cells depend on state created
   by earlier ones (`cfg`, `model`, `dataset`, `trainer`, ...).
4. After CELL 1 (dependency install) finishes, do **Runtime → Restart
   session**, then continue from CELL 2 — this is required so the
   freshly-installed packages are picked up. Don't re-run CELL 1 after
   restarting.

**Base model**: `Qwen/Qwen2.5-Coder-1.5B-Instruct`  
**Hardware**: Colab Pro, L4 GPU (~45 min)  
**Config**: `training/config/lora_1.5b.yaml`  
**Data**: `data/v1/train.jsonl` (generated by CELL 4 if it doesn't already exist)  
**Output**: `models/adapters/1.5b/` (LoRA adapter, downloaded as a zip by the last cell)

### CELL 1: Install dependencies

Installs `unsloth` (a library that patches Hugging Face's
`transformers`/`trl`/`peft` for faster, lower-memory fine-tuning) plus a
couple of small extras (`sentencepiece`, `pyyaml`), and `salestools`' own
declared dependencies (`pandas`, `statsmodels`, `scikit-learn`,
`matplotlib` — installed explicitly rather than assumed from Colab's
default image, since CELL 4 imports `salestools` transitively). Also sets
a CUDA allocator flag (`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`)
to reduce GPU memory fragmentation during training.

**After this cell finishes, restart the Colab runtime** (Runtime → Restart
session) so the newly installed packages load cleanly, then continue from
CELL 2 — don't re-run this cell after restarting.

In [ ]:
# ── CELL 1: Install dependencies ────────────────────────────────────────────
import os

# Install fine-tuning dependencies
# Let unsloth manage ALL ML library versions (peft, transformers, trl, accelerate, datasets, bitsandbytes)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q sentencepiece pyyaml

# salestools' own declared dependencies (see pyproject.toml) — installed explicitly rather
# than relying on Colab's default image happening to include them, since CELL 4 imports
# salestools transitively (via the data generator's sandboxed verifier subprocess).
!pip install -q "pandas>=2.0" "statsmodels>=0.14" "scikit-learn>=1.4" "matplotlib>=3.8"

# Prevent CUDA OOM fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### CELL 2: Imports

Imports `unsloth` first — it must be imported before
`transformers`/`trl`/`peft` so its performance patches apply. Then imports
the standard libraries used throughout the notebook (`json`, `subprocess`,
`shutil`, `time`, `pathlib.Path`, `yaml`, `torch`) plus the Hugging
Face/TRL/Unsloth classes used later (`Dataset`, `SFTTrainer`,
`TrainingArguments`, `FastLanguageModel`). Finally, checks and prints
whether a CUDA GPU is available — if this prints `CUDA available: False`,
stop and fix your Colab runtime type (Runtime → Change runtime type → GPU)
before continuing.

In [ ]:
# ── CELL 2: Imports ──────────────────────────────────────────────────────────
import unsloth  # must be first — patches trl/transformers/peft before they load

import json, os, subprocess, shutil, time
from pathlib import Path
import yaml
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### CELL 3: Paths and config

Defines all the filesystem paths the rest of the notebook uses (repo root,
config YAML, system prompt, training data, adapter output directory)
relative to `/content/salestools-analyst` — Colab's local (ephemeral) disk,
not Google Drive.

If the config file isn't already present, it (re)clones this GitHub repo
into that path — including wiping out any stale/incomplete directory left
over from a previous failed attempt, so a half-finished clone can't
silently make the rest of the notebook fail. It then loads
`training/config/lora_1.5b.yaml` (hyperparameters, LoRA settings) into the
`cfg` dict and reads the system prompt text — both are used by nearly every
cell after this one.

In [ ]:
# ── CELL 3: Paths and config ─────────────────────────────────────────────────
REPO_ROOT   = Path("/content/salestools-analyst")
CONFIG_PATH = REPO_ROOT / "training/config/lora_1.5b.yaml"
PROMPT_PATH = REPO_ROOT / "training/config/system_prompt.txt"
TRAIN_DATA  = REPO_ROOT / "data/v1/train.jsonl"
ADAPTER_OUT = REPO_ROOT / "models/adapters/1.5b"

# Clone if config file not present (handles empty-dir edge case)
if not CONFIG_PATH.exists():
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    subprocess.run(["git", "clone", "https://github.com/sandeepkesarkar/salestools-analyst.git", str(REPO_ROOT)], check=True)

ADAPTER_OUT.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
SYSTEM_PROMPT = Path(PROMPT_PATH).read_text().strip()

print("Config:", cfg)
print(f"System prompt: {SYSTEM_PROMPT[:80]}...")

### CELL 4: Generate training data (skip if exists)

If `data/v1/train.jsonl` doesn't already exist in the cloned repo, runs
`data/generator/generate.py` to synthesize 1000 verified training examples.
Each candidate example is a synthetic sales dataset plus a natural-language
question plus the `salestools` code that should answer it; a sandboxed
subprocess actually executes that code against the data and checks it
produces the expected signal before the pair is accepted — that
verification step (not the data synthesis itself) is why this cell can take
a while to run. Prints a `[N/1000] pairs generated...` progress line every
25 pairs so you can tell it's making progress.

In [ ]:
# ── CELL 4: Generate training data (skip if exists) ─────────────────────────
if not TRAIN_DATA.exists():
    subprocess.run(
        ["python", "data/generator/generate.py", "--salestools-version", "1.0.0", "--count", "1000", "--seed", "0",
         "--signal-types", "trend_up,trend_down,anomaly_spike,anomaly_drop,anomaly_contextual,segment_drag,scope_refusal",
         "--output", str(TRAIN_DATA)],
        cwd=str(REPO_ROOT), check=True
    )
else:
    print("train.jsonl already exists, skipping generation.")

### CELL 5: Load training data

Reads `train.jsonl` line by line, keeps only pairs marked `verified: true`,
and reformats each `(question, code)` pair into a single ChatML-style
training string (system prompt + user question + assistant code, wrapped in
`<|im_start|>`/`<|im_end|>` tags — the format Qwen2.5's chat template
expects). Wraps the resulting list of strings in a Hugging Face `Dataset`
object (`dataset`), which is what the trainer consumes in CELL 8/9.

In [ ]:
# ── CELL 5: Load training data ───────────────────────────────────────────────
CHATML_TEMPLATE = (
    "<|im_start|>system\n{system}<|im_end|>\n"
    "<|im_start|>user\n{user}<|im_end|>\n"
    "<|im_start|>assistant\n{assistant}<|im_end|>"
)

records = []
with open(TRAIN_DATA) as f:
    for line in f:
        pair = json.loads(line.strip())
        if not pair.get("verified", False):
            continue
        records.append({"text": CHATML_TEMPLATE.format(
            system=SYSTEM_PROMPT,
            user=pair["question"],
            assistant=pair["code"],
        )})

dataset = Dataset.from_list(records)
print(f"Loaded {len(dataset)} verified pairs")
print("Sample:\n", dataset[0]["text"][:300])

### CELL 6: Load base model with Unsloth

Downloads (on first run) and loads the base model named in
`cfg["base_model"]` (`Qwen/Qwen2.5-Coder-1.5B-Instruct`) via Unsloth's
`FastLanguageModel.from_pretrained`, in 4-bit quantized form
(`load_in_4bit=True`) — this is the "Q" in QLoRA, and is what keeps memory
usage low enough to fine-tune on a single Colab GPU. `max_seq_length` is
read from the config and caps how many tokens of context the model will
train/infer on. Prints how long loading took.

In [ ]:
# ── CELL 6: Load base model with Unsloth ─────────────────────────────────────
print(f"Loading base model {cfg['base_model']} (downloads on first run, can take a few minutes)...")
_t0 = time.time()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg["base_model"],
    max_seq_length=cfg["training"]["max_seq_length"],
    dtype=None,           # auto-detect; bf16 on Ampere+
    load_in_4bit=True,    # QLoRA: 4-bit base
)
print(f"Base model loaded in {time.time() - _t0:.0f}s.")

### CELL 7: Apply LoRA adapters

Wraps the 4-bit base model with trainable LoRA adapters via
`FastLanguageModel.get_peft_model`. Instead of updating all ~1.5B of the
base model's weights, this inserts small trainable low-rank matrices into
the attention and MLP projection layers listed in
`cfg["lora"]["target_modules"]` — only those (a small fraction of total
parameters, printed by `model.print_trainable_parameters()`) get updated
during training. `r` and `lora_alpha` control the adapter's rank/scaling;
`use_gradient_checkpointing="unsloth"` trades some compute for lower memory
use.

In [ ]:
# ── CELL 7: Apply LoRA adapters ───────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=cfg["lora"]["r"],
    target_modules=cfg["lora"]["target_modules"],
    lora_alpha=cfg["lora"]["lora_alpha"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=cfg["training"]["seed"],
)
print("LoRA applied.")
model.print_trainable_parameters()

### CELL 8: SFTTrainer setup

Builds a TRL `SFTTrainer` — the object that actually drives supervised
fine-tuning. Tokenizes the ChatML `dataset` from CELL 5 up front (you'll see
a tokenization progress bar). `TrainingArguments` wires in the
hyperparameters from `cfg["training"]` (batch size, gradient accumulation,
learning rate, epochs, scheduler, etc.); `report_to="none"` disables sending
metrics to any external experiment tracker (e.g. W&B) since none is
configured here. Doesn't start training yet — just prepares everything so
CELL 9 can.

In [ ]:
# ── CELL 8: SFTTrainer setup ─────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=cfg["training"]["max_seq_length"],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=cfg["training"]["per_device_train_batch_size"],
        gradient_accumulation_steps=cfg["training"]["gradient_accumulation_steps"],
        warmup_ratio=cfg["training"]["warmup_ratio"],
        num_train_epochs=cfg["training"]["num_train_epochs"],
        learning_rate=cfg["training"]["learning_rate"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=cfg["training"]["logging_steps"],
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type=cfg["training"]["lr_scheduler_type"],
        seed=cfg["training"]["seed"],
        output_dir=str(ADAPTER_OUT / "checkpoints"),
        save_strategy=cfg["training"]["save_strategy"],
        save_total_limit=2,
        report_to="none",
    ),
)
print("Trainer ready.")

### CELL 9: Train

Runs the actual fine-tuning loop (`trainer.train()`) — this is the
longest-running cell in the notebook (tens of minutes, depending on GPU and
`num_train_epochs`). Hugging Face's `Trainer` shows its own live progress
bar and logs the training loss every `logging_steps` steps (from config) as
it goes, so you can watch it progress in real time. Prints total wall-clock
runtime when done.

In [ ]:
# ── CELL 9: Train ─────────────────────────────────────────────────────────────
print(f"Starting training ({cfg['training']['num_train_epochs']} epochs, "
      f"logging every {cfg['training']['logging_steps']} steps)...")
trainer_stats = trainer.train()
print(f"Training complete. Runtime: {trainer_stats.metrics.get('train_runtime', 0):.0f}s")

### CELL 10: Save LoRA adapter

Saves the fine-tuned LoRA adapter weights and tokenizer to `ADAPTER_OUT`
(`models/adapters/1.5b/` under the cloned repo) using PEFT's
`save_pretrained`. This is just the small adapter, not a merged copy of the
full base model — lists the files written so you can sanity-check the save
succeeded.

In [ ]:
# ── CELL 10: Save LoRA adapter ────────────────────────────────────────────────
model.save_pretrained(str(ADAPTER_OUT))
tokenizer.save_pretrained(str(ADAPTER_OUT))
print(f"Adapter saved to {ADAPTER_OUT}")
print("Files:", list(ADAPTER_OUT.glob("*")))

### CELL 11: Download adapter

Zips the saved adapter directory and triggers a browser download via
Colab's `google.colab.files.download` — this is how you get the trained
adapter off the (ephemeral) Colab VM and onto your local machine, since
anything under `/content` is deleted when the Colab runtime
disconnects/recycles. See the "Next Steps" cell below for what to do with
it once downloaded.

In [ ]:
# ── CELL 11: Download adapter ─────────────────────────────────────────────────
from google.colab import files

shutil.make_archive("/content/adapter_1.5b", "zip", str(ADAPTER_OUT))
files.download("/content/adapter_1.5b.zip")

## Next Steps

1. The last cell downloads `adapter_1.5b.zip` to your machine — unzip it into `models/adapters/1.5b/`
2. Run `bash training/export.sh` (set `MODEL_SIZE=1.5b ADAPTER_PATH=models/adapters/1.5b/`)
3. Verify: `ollama run sales-analyst-1.5b "Is my trend going up?"`
4. Eval: `python eval/run_eval.py --model sales-analyst-1.5b --held-out data/v1/held_out.jsonl`